# 03 - Quantization Cost/Latency/Quality Tradeoff Demo

Companion notebook to `04-deployment-architecture-azure-ml-aks.md`. A conceptual model of the
cost/latency/quality tradeoff quantization introduces: comparing a synthetic "full precision" Mixtral
deployment against progressively more aggressively quantized configurations, in terms of memory
footprint, relative throughput, and a clearly-labeled *illustrative* quality-degradation estimate.

The memory-footprint math is real arithmetic (bytes-per-parameter times parameter count). The
throughput-improvement and quality-degradation figures are clearly-labeled illustrative estimates used
to show the *shape* of the tradeoff described in chapter 04 -- not measured benchmark results. No real
model, no GPU, no network calls.

## 1. Memory footprint at each precision level

Mixtral 8x7B's ~47B total parameters (chapter 01) need to be held in GPU memory at whatever precision
is being served. Bytes-per-parameter scales directly with bit-width: FP16/BF16 uses 2 bytes/parameter,
INT8 uses 1 byte/parameter, INT4 uses 0.5 bytes/parameter.

In [1]:
TOTAL_PARAMS = 47e9  # Mixtral 8x7B's real, published total parameter count (approx.)

PRECISIONS = [
    ("FP16 / BF16 (full precision)", 2.0),
    ("INT8 (8-bit quantized)", 1.0),
    ("INT4 (4-bit quantized)", 0.5),
]

def weights_memory_gb(total_params, bytes_per_param):
    return (total_params * bytes_per_param) / 1e9  # GB, using 1e9 bytes/GB for a round, illustrative unit

print("{:32s} {:>16s} {:>18s}".format("Precision", "Bytes/param", "Weights memory (GB)"))
print("-" * 70)
footprints = {}
for name, bpp in PRECISIONS:
    gb = weights_memory_gb(TOTAL_PARAMS, bpp)
    footprints[name] = gb
    print("{:32s} {:16.2f} {:18.1f}".format(name, bpp, gb))

fp16_gb = footprints["FP16 / BF16 (full precision)"]
int4_gb = footprints["INT4 (4-bit quantized)"]
print()
print("INT4 quantization cuts weights memory to {:.0f}% of FP16's footprint ({:.1f}GB vs {:.1f}GB),".format(
    int4_gb / fp16_gb * 100, int4_gb, fp16_gb))
print("directly changing which GPU instance tiers can hold the model at all (chapter 04).")


Precision                             Bytes/param Weights memory (GB)
----------------------------------------------------------------------
FP16 / BF16 (full precision)                 2.00               94.0
INT8 (8-bit quantized)                       1.00               47.0
INT4 (4-bit quantized)                       0.50               23.5

INT4 quantization cuts weights memory to 25% of FP16's footprint (23.5GB vs 94.0GB),
directly changing which GPU instance tiers can hold the model at all (chapter 04).


## 2. Relative throughput: illustrative, but grounded in a real mechanism

Lower-precision weights move less data through GPU memory bandwidth per token, which is very often the
actual inference bottleneck (not raw compute) -- so quantization frequently improves throughput, not
just memory footprint. The *specific* multiplier below is an illustrative, clearly-labeled estimate
(not a measured benchmark), used only to show the shape of the effect.

In [2]:
# Illustrative relative-throughput multipliers (NOT measured benchmark numbers) -- grounded in the real
# mechanism that lower-precision weights need less memory bandwidth to move per token.
ILLUSTRATIVE_RELATIVE_THROUGHPUT = {
    "FP16 / BF16 (full precision)": 1.00,
    "INT8 (8-bit quantized)": 1.35,
    "INT4 (4-bit quantized)": 1.70,
}

BASELINE_LATENCY_MS_PER_TOKEN = 42.0  # illustrative baseline, FP16, for a fixed reference batch/hardware config

print("{:32s} {:>22s} {:>24s}".format("Precision", "Relative throughput", "Illustrative ms/token"))
print("-" * 82)
for name, _ in PRECISIONS:
    mult = ILLUSTRATIVE_RELATIVE_THROUGHPUT[name]
    latency = BASELINE_LATENCY_MS_PER_TOKEN / mult
    print("{:32s} {:21.2f}x {:23.1f}ms".format(name, mult, latency))

print()
print("Illustrative only -- real throughput gains depend heavily on the serving framework, batch size,")
print("and hardware generation (chapter 04). The point is the DIRECTION and rough SHAPE of the effect:")
print("lower precision typically means faster, not just smaller.")


Precision                           Relative throughput    Illustrative ms/token
----------------------------------------------------------------------------------
FP16 / BF16 (full precision)                      1.00x                    42.0ms
INT8 (8-bit quantized)                            1.35x                    31.1ms
INT4 (4-bit quantized)                            1.70x                    24.7ms

Illustrative only -- real throughput gains depend heavily on the serving framework, batch size,
and hardware generation (chapter 04). The point is the DIRECTION and rough SHAPE of the effect:
lower precision typically means faster, not just smaller.


## 3. Quality degradation: illustrative estimate, and why it's not uniform

This is the real tradeoff quantization costs you, and chapter 04/08 are explicit that the degradation is
**not uniform across task types** -- it can concentrate on the harder, more nuanced parts of a task while
barely touching the easier, templated parts. We model that directly: a small overall degradation, plus a
larger, concentrated degradation on one harder category (mirroring chapter 08's quantization bug
narrative, where an 8-bit config passed in aggregate but hurt one specific, less-common taxonomy
category).

In [3]:
# Illustrative, clearly-labeled quality estimates -- NOT measured evaluation results.
# "Overall" reflects an aggregate accuracy metric across a whole domain evaluation set (chapter 05).
# "Hardest category" reflects one harder, less-common taxonomy category within that same set --
# exactly the kind of narrow regression chapter 08's Bug 1 describes an aggregate-only check would miss.

BASELINE_ACCURACY = {"overall": 0.93, "hardest_category": 0.85}

ILLUSTRATIVE_QUALITY_DELTA = {
    "FP16 / BF16 (full precision)": {"overall": 0.000, "hardest_category": 0.000},
    "INT8 (8-bit quantized)": {"overall": -0.004, "hardest_category": -0.061},
    "INT4 (4-bit quantized)": {"overall": -0.021, "hardest_category": -0.145},
}

print("{:32s} {:>18s} {:>22s}".format("Precision", "Overall accuracy", "Hardest-category accuracy"))
print("-" * 76)
for name, _ in PRECISIONS:
    delta = ILLUSTRATIVE_QUALITY_DELTA[name]
    overall = BASELINE_ACCURACY["overall"] + delta["overall"]
    hardest = BASELINE_ACCURACY["hardest_category"] + delta["hardest_category"]
    print("{:32s} {:17.1%} {:21.1%}".format(name, overall, hardest))

overall_drop_int8 = -ILLUSTRATIVE_QUALITY_DELTA["INT8 (8-bit quantized)"]["overall"]
hardest_drop_int8 = -ILLUSTRATIVE_QUALITY_DELTA["INT8 (8-bit quantized)"]["hardest_category"]
print()
print("At INT8: overall accuracy drops only {:.1%}, which would likely pass an aggregate-only check.".format(overall_drop_int8))
print("But the hardest category alone drops {:.1%} -- {:.1f}x the aggregate drop -- exactly the kind of".format(
    hardest_drop_int8, hardest_drop_int8 / overall_drop_int8))
print("narrow, hidden regression chapter 08's Bug 1 describes, and exactly why chapter 05's domain")
print("evaluation set is scored PER-CATEGORY, not just in aggregate.")

assert hardest_drop_int8 > overall_drop_int8 * 3, "The hardest category's degradation should dwarf the aggregate figure, illustrating why per-category scoring matters."


Precision                          Overall accuracy Hardest-category accuracy
----------------------------------------------------------------------------
FP16 / BF16 (full precision)                 93.0%                 85.0%
INT8 (8-bit quantized)                       92.6%                 78.9%
INT4 (4-bit quantized)                       90.9%                 70.5%

At INT8: overall accuracy drops only 0.4%, which would likely pass an aggregate-only check.
But the hardest category alone drops 6.1% -- 15.2x the aggregate drop -- exactly the kind of
narrow, hidden regression chapter 08's Bug 1 describes, and exactly why chapter 05's domain
evaluation set is scored PER-CATEGORY, not just in aggregate.


## 4. Putting it together: the cost/latency/quality curve

A simple side-by-side view of all three axes at once -- this is the tradeoff surface chapter 04 argues
should be navigated deliberately, measured against the real domain evaluation set at each candidate
quantization level, rather than defaulted into as a blanket cost-cutting switch.

In [4]:
print("{:32s} {:>14s} {:>16s} {:>12s} {:>16s}".format(
    "Precision", "Memory (GB)", "Rel. throughput", "Overall acc.", "Hardest-cat acc."))
print("-" * 94)
for name, _ in PRECISIONS:
    gb = footprints[name]
    mult = ILLUSTRATIVE_RELATIVE_THROUGHPUT[name]
    delta = ILLUSTRATIVE_QUALITY_DELTA[name]
    overall = BASELINE_ACCURACY["overall"] + delta["overall"]
    hardest = BASELINE_ACCURACY["hardest_category"] + delta["hardest_category"]
    print("{:32s} {:14.1f} {:15.2f}x {:11.1%} {:15.1%}".format(name, gb, mult, overall, hardest))

print()
print("No single row is universally 'best' -- INT4 is cheapest and fastest but costs the most quality,")
print("concentrated on the hardest category; FP16 is safest on quality but most expensive. Chapter 04's")
print("point: choosing a point on this curve is a measured decision against the domain eval set (chapter")
print("05), not a default flipped for cost savings without checking what it costs on the hard cases.")


Precision                           Memory (GB)  Rel. throughput Overall acc. Hardest-cat acc.
----------------------------------------------------------------------------------------------
FP16 / BF16 (full precision)               94.0            1.00x       93.0%           85.0%
INT8 (8-bit quantized)                     47.0            1.35x       92.6%           78.9%
INT4 (4-bit quantized)                     23.5            1.70x       90.9%           70.5%

No single row is universally 'best' -- INT4 is cheapest and fastest but costs the most quality,
concentrated on the hardest category; FP16 is safest on quality but most expensive. Chapter 04's
point: choosing a point on this curve is a measured decision against the domain eval set (chapter
05), not a default flipped for cost savings without checking what it costs on the hard cases.


## Summary

| Section | Computes | Matches |
|---|---|---|
| 1 | Real memory-footprint arithmetic (bytes/param x total params) at 3 precision levels | Chapter 04's GPU-sizing discussion |
| 2 | Illustrative relative-throughput multipliers, grounded in the real memory-bandwidth mechanism | Chapter 04's quantization-as-latency-lever discussion |
| 3 | Illustrative, non-uniform quality degradation -- concentrated on a harder category | Chapter 08's Bug 1 (a quantization regression an aggregate-only check would miss) |
| 4 | All three axes side by side as a tradeoff surface | Chapter 04's "measure against the domain eval set at each candidate level" conclusion |

Every illustrative figure in this notebook is explicitly labeled as such -- the real, load-bearing points
are the memory arithmetic (section 1, genuine) and the *shape* of the tradeoff (sections 2-4): lower
precision is cheaper and faster, but the quality cost is real and non-uniform, which is why it has to be
measured, not assumed.